[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IyadSultan/CCI/blob/main/session3/student/Lab1_OpenAI_API_Fundamentals_Student.ipynb)

# Lab 1: OpenAI API Fundamentals
## CCI Session 3 — Data Science Foundations

**Duration:** 15 minutes
**Mode:** Guided step-by-step

### Clinical Scenario
> You are building the first component of a clinical AI pipeline at KHCC. Your task is to connect to the OpenAI API from Python, understand how tokens and costs work, and apply different prompting techniques to classify clinical text.

### Objective
By the end of this lab, you will:
- Make API calls to GPT models from Python
- Count tokens and estimate costs
- Apply zero-shot, few-shot, and chain-of-thought prompting

## Setup — Install and Import

In [4]:
# Run this cell first to install required packages
!pip install -q openai tiktoken

import openai
from openai import OpenAI
import tiktoken
import json
import os # operating system

if not os.path.exists('/content/CCI'):
    !git clone https://github.com/IyadSultan/CCI.git /content/CCI
os.chdir('/content/CCI/session3/student')

Cloning into '/content/CCI'...
remote: Enumerating objects: 313, done.
remote: Counting objects: 100% (313/313), done.
remote: Compressing objects: 100% (212/212), done.
remote: Total 313 (delta 160), reused 201 (delta 96), pack-reused 0 (from 0)
Receiving objects: 100% (313/313), 1.91 MiB | 24.40 MiB/s, done.
Resolving deltas: 100% (160/160), done.


## Cell 1 — API Key Setup
Your OpenAI API key should be stored in **Colab Secrets** (click the key icon in the left sidebar) under the name `OPENAI_API_KEY`.

In [12]:
from openai.lib.azure import API_KEY_SENTINEL
# === CELL 1: API KEY ===
# Make sure your API key is stored in Colab Secrets (click the key icon in the left sidebar)
# under the name: OPENAI_API_KEY
from google.colab import userdata

from google.colab import userdata
api_key = userdata.get('OPENAI_API_KEY').strip()
client = OpenAI(api_key=api_key)
print("Client created successfully!")

Client created successfully!


## Cell 2 — Your First API Call
Make a simple chat completion call. Ask the model to classify a clinical note as one of:
**Discharge Summary, Progress Note, Consultation Note, Pathology Report.**

In [13]:
from openai import OpenAI
client = OpenAI(api_key=api_key)


clinical_note = """
Patient was seen in the oncology clinic for follow-up after completing
4 cycles of AC chemotherapy for Stage IIA breast cancer. No new complaints.
Physical exam unremarkable. Plan to start paclitaxel weekly x12.
"""

response = client.chat.completions.create(
    model="gpt-4o-mini",
    temperature=0.0,
    messages=[
        {
            "role": "system",
            "content": "You are a clinical document classifier. Classify the note as one of: Discharge Summary, Progress Note, Consultation Note, Pathology Report. Return only the classification."
        },
        {
            "role": "user",
            "content": clinical_note
        }
    ]
)

# Extract and print the response
print(response.choices[0].message.content)

Progress Note


## Cell 3 — Token Counting with tiktoken
Count tokens BEFORE sending to estimate costs and check context limits.

In [15]:
# === CELL 3: TOKEN COUNTING ===

# Create a tiktoken encoder for the model
encoding = tiktoken.encoding_for_model("gpt-4o-mini")

# Count tokens in the clinical_note
tokens = encoding.encode(clinical_note)

# Print: number of tokens, first 10 token IDs, and decoded first 10 tokens
print(f"Number of tokens in clinical_note: {len(tokens)}")
print(f"First 10 token IDs: {tokens[:10]}")
print(f"Decoded first 10 tokens: {encoding.decode(tokens[:10])}")

# Calculate estimated cost
# GPT-4o-mini pricing: $0.15 per 1M input tokens, $0.60 per 1M output tokens
input_tokens = len(tokens)
output_tokens = 20 # As specified in the prompt hint

input_cost_per_million = 0.15
output_cost_per_million = 0.60

estimated_input_cost = (input_tokens / 1_000_000) * input_cost_per_million
estimated_output_cost = (output_tokens / 1_000_000) * output_cost_per_million
estimated_total_cost = estimated_input_cost + estimated_output_cost

print(f"\nEstimated input tokens: {input_tokens}")
print(f"Estimated output tokens: {output_tokens}")
print(f"Estimated total cost: ${estimated_total_cost:.6f}")

Number of tokens in clinical_note: 47
First 10 token IDs: [198, 33235, 673, 6177, 306, 290, 173769, 37668, 395, 2622]
Decoded first 10 tokens: 
Patient was seen in the oncology clinic for follow

Estimated input tokens: 47
Estimated output tokens: 20
Estimated total cost: $0.000019


## Cell 4 — Temperature Experiment
Run the same prompt 3 times with `temperature=0.0`, then 3 times with `temperature=1.5`.
Compare consistency.

In [16]:
# === CELL 4: TEMPERATURE EXPERIMENT ===
prompt = "List 3 possible side effects of cisplatin chemotherapy."

# Run the prompt 3 times with temperature=0.0
deterministic_results = []
for i in range(3):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0.0,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    deterministic_results.append(response.choices[0].message.content)

# Run the prompt 3 times with temperature=1.5
random_results = []
for i in range(3):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=1.5,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    random_results.append(response.choices[0].message.content)

# Print all results and compare
print("\n--- Deterministic Results (temperature=0.0) ---")
for i, result in enumerate(deterministic_results):
    print(f"Run {i+1}:\n{result}\n")

print("\n--- Random Results (temperature=1.5) ---")
for i, result in enumerate(random_results):
    print(f"Run {i+1}:\n{result}\n")

# Comparison commentary
print("\nComparison:")
print(f"Are the deterministic results identical? {len(set(deterministic_results)) == 1}")
print(f"Are the random results different? {len(set(random_results)) > 1}")


--- Deterministic Results (temperature=0.0) ---
Run 1:
Cisplatin chemotherapy can cause a variety of side effects. Three possible side effects include:

1. **Nausea and Vomiting**: Cisplatin is known to cause significant nausea and vomiting, which can occur shortly after administration or may be delayed for several days.

2. **Kidney Toxicity**: Cisplatin can lead to nephrotoxicity, which may result in decreased kidney function. Patients may require hydration and monitoring of kidney function during treatment.

3. **Hearing Loss**: Cisplatin can cause ototoxicity, leading to hearing loss or tinnitus (ringing in the ears), particularly with higher doses or prolonged use.

It's important for patients receiving cisplatin to be monitored closely for these and other potential side effects.

Run 2:
Cisplatin chemotherapy can cause a variety of side effects. Three possible side effects include:

1. **Nausea and Vomiting**: Cisplatin is known to cause significant nausea and vomiting, which ca

## Cell 5 — Prompting Techniques
Apply **zero-shot**, **few-shot**, and **chain-of-thought** to the same clinical task:
extracting the cancer type from a clinical note.

In [17]:
# === CELL 5: PROMPTING TECHNIQUES ===
test_note = "Patient is a 62-year-old male presenting with a 3cm mass in the sigmoid colon, biopsy-confirmed adenocarcinoma. CEA elevated at 8.2. CT shows no distant metastases. Plan: surgical resection followed by adjuvant FOLFOX."

print("--- Zero-Shot Prompting ---")
# --- Zero-Shot ---
# Simple prompt - just ask "What is the cancer type?"
zero_shot_response = client.chat.completions.create(
    model="gpt-4o-mini",
    temperature=0.0,
    messages=[
        {
            "role": "system",
            "content": "You are a clinical assistant. Extract the cancer type from the clinical note. Return only the cancer type."
        },
        {
            "role": "user",
            "content": test_note
        }
    ]
)
print(f"Zero-Shot: {zero_shot_response.choices[0].message.content}\n")

print("--- Few-Shot Prompting ---")
# --- Few-Shot ---
# Provide 2 examples first, then ask about the test note
few_shot_response = client.chat.completions.create(
    model="gpt-4o-mini",
    temperature=0.0,
    messages=[
        {
            "role": "system",
            "content": "You are a clinical assistant. Extract the cancer type from the clinical note based on the examples provided. Return only the cancer type."
        },
        {
            "role": "user",
            "content": "Note: 52F with ER+ invasive ductal carcinoma of the left breast"
        },
        {
            "role": "assistant",
            "content": "Breast cancer - Invasive ductal carcinoma"
        },
        {
            "role": "user",
            "content": "Note: 71M diagnosed with small cell lung cancer, extensive stage"
        },
        {
            "role": "assistant",
            "content": "Lung cancer - Small cell"
        },
        {
            "role": "user",
            "content": f"Note: {test_note}"
        }
    ]
)
print(f"Few-Shot: {few_shot_response.choices[0].message.content}\n")

print("--- Chain-of-Thought Prompting ---")
# --- Chain-of-Thought ---
# Ask the model to "Think step by step: 1) Identify the organ, 2) Identify the histology, 3) State the cancer type"
chain_of_thought_response = client.chat.completions.create(
    model="gpt-4o-mini",
    temperature=0.0,
    messages=[
        {
            "role": "system",
            "content": "You are a clinical assistant. Extract the cancer type from the clinical note. Think step by step: 1) Identify the organ, 2) Identify the histology, 3) State the cancer type. Provide the final cancer type at the end."
        },
        {
            "role": "user",
            "content": test_note
        }
    ]
)
print(f"Chain-of-Thought: {chain_of_thought_response.choices[0].message.content}\n")

--- Zero-Shot Prompting ---
Zero-Shot: Adenocarcinoma of the sigmoid colon

--- Few-Shot Prompting ---
Few-Shot: Colon cancer - Adenocarcinoma

--- Chain-of-Thought Prompting ---
Chain-of-Thought: 1) Identify the organ: The mass is located in the sigmoid colon.
2) Identify the histology: The biopsy confirmed adenocarcinoma.
3) State the cancer type: The cancer type is colorectal cancer.

Final cancer type: Colorectal adenocarcinoma.



## Cell 6 — Streaming Responses
For long outputs, stream tokens as they arrive.

In [18]:
# === CELL 6: STREAMING ===
# TODO: Make a streaming API call (stream=True)
# Ask: "Write a brief patient education summary about managing nausea during chemotherapy"
# Print each chunk as it arrives

# Hint:
# stream = client.chat.completions.create(..., stream=True)
# for chunk in stream:
#     if chunk.choices[0].delta.content:
#         print(chunk.choices[0].delta.content, end="")

print("Streaming response for patient education summary:")

stream = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "user",
            "content": "Write a brief patient education summary about managing nausea during chemotherapy"
        }
    ],
    stream=True,
)

for chunk in stream:
    if chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="")
print("\n") # Add a newline at the end for clean output

Streaming response for patient education summary:
### Managing Nausea During Chemotherapy: Patient Education Summary

Chemotherapy can often lead to nausea and vomiting, but there are effective strategies to help manage these symptoms. Here are some key points to consider:

**1. Medications:**
   - **Antiemetics:** Your doctor may prescribe medications specifically to prevent or treat nausea. Common drugs include ondansetron, metoclopramide, and dexamethasone. Take these as directed, and don’t hesitate to discuss any side effects with your healthcare provider.

**2. Dietary Tips:**
   - **Small, Frequent Meals:** Eating smaller meals throughout the day instead of large ones can help reduce nausea.
   - **Bland Foods:** Opt for easily digestible foods such as toast, crackers, bananas, and rice.
   - **Stay Hydrated:** Sip clear liquids like water, broth, or ginger ale to stay hydrated. Avoid strong smells and rich, greasy foods that can trigger nausea.

**3. Lifestyle Modifications:**
 

## Stretch Challenge
Build a function `classify_note(note_text)` that takes any clinical note, counts tokens,
makes an API call, and returns both the classification and the cost estimate.

In [ ]:
# === STRETCH CHALLENGE ===
# TODO: Build the classify_note function

def classify_note(note_text):
    """
    Classify a clinical note and return the classification with cost estimate.

    Args:
        note_text (str): The clinical note text

    Returns:
        dict: {"classification": str, "input_tokens": int, "output_tokens": int, "estimated_cost_usd": float}
    """
    pass  # TODO: Implement this function

## Expected Outputs

| Cell | What You Should See |
|------|--------------------|
| Cell 2 | `Progress Note` |
| Cell 3 | Token count ~45-55, cost < $0.001 |
| Cell 4 | temperature=0.0 gives identical outputs; temperature=1.5 gives varied outputs |
| Cell 5 | All three techniques identify "Colorectal cancer - Adenocarcinoma" (CoT gives reasoning) |
| Cell 6 | Tokens stream in real-time |

---

### KHCC Connection
This lab mirrors the first step in the **AIDI clinical AI pipeline**: connecting to LLM APIs
from Python, understanding costs at scale, and choosing the right prompting strategy for
clinical NLP tasks like note classification and entity extraction.